Neural Training

In [1]:
print('--- STEP 0: INSTALLING DEPENDENCIES ---')
# Install compatible versions of the required libraries for CUDA 13
!pip uninstall -y trl peft accelerate transformers

!pip install -q \
transformers==4.41.2 \
trl==0.9.6 \
peft==0.11.1 \
accelerate==0.31.0

--- STEP 0: INSTALLING DEPENDENCIES ---
Found existing installation: trl 0.9.6
Uninstalling trl-0.9.6:
  Successfully uninstalled trl-0.9.6
Found existing installation: peft 0.11.1
Uninstalling peft-0.11.1:
  Successfully uninstalled peft-0.11.1
Found existing installation: accelerate 0.31.0
Uninstalling accelerate-0.31.0:
  Successfully uninstalled accelerate-0.31.0
Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2


In [2]:
!pip uninstall -y numpy pandas
!pip install -q numpy==1.26.4 pandas==2.2.2

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 56.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which i

In [3]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [4]:
print("--- STEP 1: MERGING DATASETS ---")
# Load the cleaned datasets you created earlier
codex_df = pd.read_csv("codex_clean.csv")
mqa_df = pd.read_csv("unified_mqa_train.csv")

--- STEP 1: MERGING DATASETS ---


In [5]:
master_train_df = pd.concat([codex_df, mqa_df], ignore_index=True)
master_train_df = master_train_df.sample(frac=1, random_state=42).reset_index(drop=True) # Shuffle
dataset = Dataset.from_pandas(master_train_df)


In [6]:
model_id = "microsoft/Phi-3-mini-4k-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
def format_prompt(row):
    """The strict format we force the AI to learn."""
    prompt = f"### Instruction: Write Python code to solve the math problem. Store the answer in 'result'.\n"
    prompt += f"### Question:\n{row['question']}\n"
    prompt += f"### Code:\n{row['code']}"
    return {"text": prompt + tokenizer.eos_token}

formatted_dataset = dataset.map(format_prompt)

Map:   0%|          | 0/12000 [00:00<?, ? examples/s]

In [8]:
print("--- STEP 2: LOADING MODEL IN 4-BIT ---")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

--- STEP 2: LOADING MODEL IN 4-BIT ---


In [9]:
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

In [10]:
print("--- STEP 3: APPLYING QLoRA ---")
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)


--- STEP 3: APPLYING QLoRA ---


In [11]:
print("--- STEP 4: TRAINING ---")
training_args = TrainingArguments(
    output_dir="./phi3-math-agent",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=500, # Use max_steps for a quick test, or change to num_train_epochs=1 for full training
    fp16=True,
    optim="paged_adamw_8bit"
)


--- STEP 4: TRAINING ---


In [1]:
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()


NameError: name 'SFTTrainer' is not defined

In [ ]:
print("--- STEP 5: SAVING ---")
trainer.model.save_pretrained("phi3-neuro-symbolic-adapter")
tokenizer.save_pretrained("phi3-neuro-symbolic-adapter")